# 1. Visible reasoning is not hidden reasoning

This notebook introduces small language models (SLMs), tokenization, and visible step-by-step rationales. It deliberately separates generated text from claims about internal representations.

**Prerequisites:** linear algebra, probability, and basic PyTorch/Transformers familiarity. Read [the reference list](../tutorials/REFERENCES.md) before the Jacobian section.

## Interpretation boundary

A model can be prompted to produce a rationale. That rationale is observable generated text, not verified access to private hidden reasoning. Later notebooks inspect representation probes and retain the same caution.

In [ ]:
# CPU is the default profile for this notebook. Select a GPU only for faster generation.
import sys
from pathlib import Path
repo = Path('/content/slm-jspace-viewer')
if not repo.exists():
    !git clone --branch feature/colab-jspace-tutorial --single-branch https://github.com/prithvimk/slm-jspace-viewer.git {repo}
%cd /content/slm-jspace-viewer
!git fetch origin feature/colab-jspace-tutorial
!git checkout feature/colab-jspace-tutorial
# Preserve Colab's driver-matched PyTorch/CUDA build.
!{sys.executable} -m pip -q install uv
!{sys.executable} scripts/build_colab_requirements.py
!{sys.executable} -m pip -q install -r /tmp/slm-jspace-colab-requirements.txt
!{sys.executable} -m pip -q install --no-deps -e .
from tutorials.lib.colab import report_runtime
report_runtime()


In [ ]:
import ipywidgets as widgets
import plotly.express as px
from IPython.display import Markdown, display
from tutorials.lib.models import load_tutorial_model, render_and_generate

# Qwen 0.5B is public; no Hugging Face login is needed.
tutorial_model = load_tutorial_model()
display(Markdown(f'Loaded **{tutorial_model.model_id}**.'))

In [ ]:
prompt = widgets.Textarea(value='Solve carefully: If a train travels 60 km in 1.5 hours, what is its speed?', description='Prompt:', layout=widgets.Layout(width='95%', height='90px'))
seed = widgets.IntSlider(value=42, min=0, max=100, description='Seed:')
length = widgets.IntSlider(value=48, min=8, max=96, step=8, description='Max tokens:')
run = widgets.Button(description='Generate visible rationale', button_style='primary')
output = widgets.Output()

def generate(_):
    with output:
        output.clear_output(wait=True)
        text, tokens = render_and_generate(tutorial_model, prompt.value, length.value, seed.value)
        display(Markdown('### Observable output'))
        print(text)
        figure = px.bar(x=list(range(len(tokens))), y=[len(token) for token in tokens], labels={'x':'generated/input token index','y':'decoded token length'}, title='How does this tokenizer segment the visible trace?')
        figure.show()

run.on_click(generate)
display(widgets.VBox([prompt, widgets.HBox([seed, length, run]), output]))

## Takeaway

You can inspect tokenized generated rationales, but this does not establish what representations were causally used internally. The next notebook starts with residual-stream measurements rather than treating the rationale as ground truth.